# TED Talks Transcript Extractor

TODO: Description

### 1. Dependencies

In [12]:
%pip install --quiet requests beautifulsoup4 pandas lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1
[notice] To update, run: C:\Users\estep\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [13]:
import json
import re
import time
from typing import Optional

import pandas as pd
import requests
from bs4 import BeautifulSoup

### 2. Settings

In [14]:
TARGET_LANGUAGES = ["zh", "en", "fi", "fr", "he", "ja", "pl"]
CHINESE_FALLBACKS = ["zh", "zh-cn", "zh-tw"]

CANDIDATES_PATH = "ted_candidates.csv"
OUTPUT_PATH = "ted_talks_transcripts.tsv"

POLITE_DELAY = 0.5

GRAPHQL_URL = "https://www.ted.com/graphql"

UA = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
)

HEADERS = {
    "User-Agent": UA,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

GRAPHQL_HEADERS = {
    "User-Agent": UA,
    "Accept": "application/json",
    "Content-Type": "application/json",
    "Origin": "https://www.ted.com",
    "Referer": "https://www.ted.com/",
}

session = requests.Session()
session.headers.update(HEADERS)

### 3. Fetching

In [15]:
def fetch_talk_metadata(slug: str) -> dict:
    url = f"https://www.ted.com/talks/{slug}"
    resp = session.get(url, timeout=30)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "lxml")
    tag = soup.find("script", id="__NEXT_DATA__")
    if tag is None or not tag.string:
        raise RuntimeError(f"__NEXT_DATA__ not found on {url}")

    data = json.loads(tag.string)
    page_props = data.get("props", {}).get("pageProps", {})

    video_data = page_props.get("videoData") or {}
    if not video_data and "video" in page_props:
        video_data = page_props["video"]

    player_data_raw = video_data.get("playerData")
    player_data = {}
    if isinstance(player_data_raw, str):
        try:
            player_data = json.loads(player_data_raw)
        except json.JSONDecodeError:
            player_data = {}
    elif isinstance(player_data_raw, dict):
        player_data = player_data_raw

    languages = []
    for entry in player_data.get("languages", []) or []:
        if isinstance(entry, dict):
            code = entry.get("languageCode") or entry.get("language_code")
            if code:
                languages.append(code)

    video_id = (
        str(video_data.get("id") or "")
        or str(player_data.get("id") or "")
        or str(player_data.get("videoId") or "")
    )

    speakers = []
    for s in (video_data.get("speakers") or []):
        if not s:
            continue
        if isinstance(s, str):
            speakers.append(s)
        elif isinstance(s, dict):
            name = (
                s.get("fullName")
                or f"{s.get('firstname', '')} {s.get('lastname', '')}".strip()
            )
            if name:
                speakers.append(name)

    video_type_raw = video_data.get("videoType")
    if isinstance(video_type_raw, dict):
        video_type = video_type_raw.get("name") or video_type_raw.get("title") or ""
    elif isinstance(video_type_raw, str):
        video_type = video_type_raw
    else:
        video_type = ""

    event_raw = video_data.get("eventName") or video_data.get("event") or ""
    if isinstance(event_raw, dict):
        event_raw = event_raw.get("name") or event_raw.get("title") or ""
    elif not isinstance(event_raw, str):
        event_raw = ""

    meta = {
        "slug": slug,
        "url": url,
        "video_id": video_id,
        "title": video_data.get("title") or player_data.get("title", ""),
        "description": video_data.get("description")
            or player_data.get("description", ""),
        "recorded_on": video_data.get("recordedOn")
            or player_data.get("recorded_at", ""),
        "published_at": video_data.get("publishedAt")
            or player_data.get("published_at", ""),
        "speakers": speakers,
        "available_languages": languages,
        "videoType": video_type,
        "event": event_raw,
    }
    return meta

In [16]:
slugs = pd.read_csv(CANDIDATES_PATH)["slug"].tolist()
print(f"Wczytano {len(slugs)} slugow z {CANDIDATES_PATH}")

Wczytano 282 slugow z ted_candidates.csv


### 4. Extracting

In [17]:
def _build_query(video_id: str, language: str) -> str:
    return (
        '{ translation(videoId:"' + video_id + '", language:"' + language + '") '
        '{ paragraphs { cues { text } } } }'
    )

def _graphql_transcript(video_id: str, language: str, timeout: int = 30) -> Optional[str]:
    payload = {"operationName": None, "query": _build_query(video_id, language)}
    resp = session.post(
        GRAPHQL_URL,
        json=payload,
        headers=GRAPHQL_HEADERS,
        timeout=timeout,
    )
    if resp.status_code != 200:
        print(f"  [{language}] HTTP {resp.status_code}")
        return None

    try:
        body = resp.json()
    except ValueError:
        print(f"  [{language}] invalid JSON response")
        return None

    translation = (body.get("data") or {}).get("translation")
    if not translation:
        return None

    lines = []
    for paragraph in translation.get("paragraphs") or []:
        cues = paragraph.get("cues") or []
        text = " ".join((c.get("text") or "").strip() for c in cues if c.get("text"))
        text = re.sub(r"\s+", " ", text).strip()
        if text:
            lines.append(text)
    return "\n".join(lines) if lines else None


def fetch_transcript(video_id: str, language: str, polite_delay: float = 0.5) -> Optional[str]:
    candidates = CHINESE_FALLBACKS if language == "zh" else [language]
    for lang_code in candidates:
        transcript = _graphql_transcript(video_id, lang_code)
        if transcript:
            time.sleep(polite_delay)
            return transcript
        time.sleep(polite_delay)
    return None

In [18]:
rows = []
total = len(slugs)

for i, slug in enumerate(slugs, 1):
    print(f" [{i}/{total}] {slug}")
    try:
        meta = fetch_talk_metadata(slug)
    except Exception as exc:
        print(f"  blad metadanych: {exc}")
        continue

    transcripts = {}
    for lang in TARGET_LANGUAGES:
        text = fetch_transcript(meta["video_id"], lang)
        transcripts[lang] = text
        status = f"{len(text):,} chars" if text else "brak"
        print(f"  {lang}: {status}")

    rows.append({
        "slug": slug,
        "Ted talk title": meta["title"],
        "description": meta["description"],
        **transcripts,
    })

print(f" Zebrano {len(rows)} wierszy")

 [1/282] adam_grant_how_to_stop_languishing_and_start_finding_flow
  zh: 5,113 chars
  en: 13,505 chars
  fi: 12,389 chars
  fr: 14,210 chars
  he: 10,817 chars
  ja: 6,358 chars
  pl: 12,911 chars
 [2/282] molly_wright_how_every_child_can_thrive_by_five
  zh: 1,569 chars
  en: 4,231 chars
  fi: 4,297 chars
  fr: 4,930 chars
  he: 3,516 chars
  ja: 2,098 chars
  pl: 4,399 chars
 [3/282] andrea_berchowitz_the_link_between_menopause_and_gender_inequity_at_work
  zh: 2,425 chars
  en: 7,804 chars
  fi: 7,112 chars
  fr: 8,829 chars
  he: 6,303 chars
  ja: 3,528 chars
  pl: 7,693 chars
 [4/282] wendy_de_la_rosa_why_talking_to_your_friends_can_help_you_save_money
  zh: 1,096 chars
  en: 3,609 chars
  fi: 3,427 chars
  fr: 3,836 chars
  he: 2,864 chars
  ja: 1,463 chars
  pl: 3,300 chars
 [5/282] elif_shafak_if_trees_could_speak
  zh: 1,014 chars
  en: 2,678 chars
  fi: 2,634 chars
  fr: 3,147 chars
  he: 2,136 chars
  ja: 1,050 chars
  pl: 2,609 chars
 [6/282] ariel_waldman_the_invisible_li

KeyboardInterrupt: 

### 5. Saving

In [19]:
df = pd.DataFrame(rows)
core_cols = ["slug", "Ted talk title", "description", *TARGET_LANGUAGES]
extra_cols = [c for c in df.columns if c not in core_cols]
df = df[core_cols + extra_cols]

In [20]:
df

,slug,Ted talk title,description,zh,en,fi,fr,he,ja,pl
0,adam_grant_how_to_stop_languishing_and_start_f...,How to stop languishing and start finding flow,"Have you found yourself staying up late, joyle...",我知道大家都有 长长的待办事项列表， 但是我讨厌浪费时间做这些 所以我有个“不办”事项列表。...,"I know you all have long to-do lists, but I ha...","Teillä on pitkiä to do -listoja, mutta vihaan ...",Vous avez tous de longues listes de choses à f...,"אני יודע שלכולכם יש רשימות ארוכות של מטלות, אב...",みんな長い「やることリスト」を 持っていると思いますが 私は時間を無駄にしたくないので 「や...,"Wiem, że wszyscy mają listy rzeczy do zrobieni..."
1,molly_wright_how_every_child_can_thrive_by_five,How every child can thrive by five,"""What if I was to tell you that a game of peek...",（小朋友笑聲）\n如果我同你講 伏匿匿呢個遊戲可以改變世界 你係唔係覺得好荒謬？ 我今日就係...,[Baby cooing]\nWhat if I was to tell you that ...,"[Vauva jokeltaa]\nMitä jos kertoisin teille, e...",[Gazouilli d’un bébé]\nEt si je vous disais qu...,[תינוק ממלמל]\nמה אם הייתי אומרת לכם שמשחק “קו...,[赤ちゃんの声]\nもし「いないいないばぁ」で 世界を変えられると言われたら どう思いますか...,(Gaworzenie dziecka)\nA gdybym wam powiedziała...
2,andrea_berchowitz_the_link_between_menopause_a...,The link between menopause and gender inequity...,"Hot flashes, joint pain, anxiety, depression, ...",在全美五百强公司里，\n只有 42 位女性CEO。 在别的国家，数据也是类似的， 有的地方还...,"Of America's 500 largest companies, only 42 ha...",Amerikan 500 suurimmasta yhtiöstä 42:ssa on na...,Seules 42 des 500 plus grandes entreprises des...,"באמריקה, מתוך 500 החברות הגדולות, רק ל-41 יש מ...",アメリカの大企業500社の内 女性CEOはたったの42人しかいません 他の国に目を向けると ...,"W 500 największych amerykańskich firmach, zale..."
3,wendy_de_la_rosa_why_talking_to_your_friends_c...,Why talking to your friends can help you save ...,What convinced British citizens to send in the...,改变人们的理财行为很难， 但也是可以做到的。\n[你的钱与你的观念---Wendy De L...,Changing people's financial behavior is diffic...,Ihmisten taloudellisen käytöksen muuttaminen o...,Changer le comportement financier des gens est...,"לשנות התנהגויות כלכליות של אנשים זה קשה, אבל ז...",人の金融行動を変えるのは 難しいです でも 可能です ［あなたのお金と心 ウェンディ・デ・ラ...,"Trudno zmienić zachowania finansowe, ale da si..."
4,elif_shafak_if_trees_could_speak,If trees could speak,How do we tell stories of humanity and nature ...,人類通常當樹係空氣， 每日佢地都會行過我哋身邊。 佢地係我哋嘅身影下， 坐低休息、訓教、 野...,Humans do not see trees. They walk by us every...,Ihmiset eivät näe puita. He kulkevat ohitsemme...,Les humains ne voient pas les arbres. Ils pass...,בני אדם לא רואים עצים. הם הולכים לידנו כל יום....,人間は木を見ない 毎日私たちのそばを 通りすぎる 私たちの陰で座り 眠り 煙草を吸い ピクニ...,Ludzie nie widzą drzew. Mijają nas każdego dni...
5,ariel_waldman_the_invisible_life_hidden_beneat...,The invisible life hidden beneath Antarctica's...,"In this tour of the microscopic world, explore...",你能猜到这是什么吗？ 你相信有个地方的生物 都是由玻璃做成的吗？ 你相信有些生命体我们肉眼看...,Can you guess what this is? What if I told you...,Arvaatteko mikä tämä on? Mitä jos kertoisin te...,Pouvez-vous deviner ce que c'est ? Et si je vo...,אתם יכולים לנחש מה זה? מה אם הייתי אומרת לכם ש...,これが何か分かりますか？ ガラスでできた生き物のいる場所が あると言ったら どう思いますか？...,"Zgadniecie, co to jest? Co, gdybym powiedziała..."


In [21]:
df.to_csv(OUTPUT_PATH, sep="\t", index=False)
print(f"Saved {len(df)} row(s) -> {OUTPUT_PATH}")

Saved 6 row(s) -> ted_talks_transcripts.tsv
